# pi0 policy server (`--backend pi0`)

Serves a local, torch.compile-free copy of the pi0 SO-101 fine-tune from the Colab-hosted `lerobot.async_inference` `PolicyServer` (lerobot 0.6.1) over gRPC, through an ngrok TCP tunnel.

| | |
|---|---|
| Lane | gRPC, stock lerobot 0.6.1 PolicyServer |
| Port → tunnel | 5173 → ngrok TCP |
| GPU runtime | L4 |
| Gated HF licences | [google/paligemma-3b-pt-224](https://huggingface.co/google/paligemma-3b-pt-224): accept with the token's account first |
| Sign-offs before running | none |
| Default checkpoint | `amindt/pi0_so101_pickandplace_redblue` |
| Server log | `/content/policy_server_pi0.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Gated licence check

Fails now, with the repo name, instead of minutes later inside the server's model load.

In [ ]:
from huggingface_hub import hf_hub_download

GATED_REPOS = ['google/paligemma-3b-pt-224']
for repo in GATED_REPOS:
    try:
        hf_hub_download(repo, 'config.json')
    except Exception as exc:
        raise RuntimeError(
            f'{repo}: {type(exc).__name__}: {exc}. The HF token cannot read this gated repo: '
            f'accept the licence at https://huggingface.co/{repo} with the token\'s account, '
            'then re-run this cell.'
        ) from exc
    print(f'{repo}: gated access OK')

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## pi0 (`--backend pi0`)

**Checkpoint:** `amindt/pi0_so101_pickandplace_redblue` (`backend_configs/pi0.json`), an SO-101
joint-space pi0 fine-tune of `lerobot/pi0_base` (dataset `amindt/so101_pickandplace_redblue`).
Inputs: `observation.state` [6], `observation.images.camera1` + `camera2` (480x640); output
action [6]. The `camera_map` sends the wrist frame as `camera1` and the left AR0144 overhead
half as `camera2`; the training dataset's `camera2` viewpoint is not documented, so that is an
assumption to check on hardware.

pi0 builds on the gated `google/paligemma-3b-pt-224`: accept its licence with the account
whose token the token cell logs in with. The gated licence check above fails early otherwise.
This notebook serves the checkpoint from `/content/ckpt/pi0_so101_nocompile` (see the pi0
section after the installs).

## Install lerobot 0.6.1 and pin transformers

Adds `grpcio` — already audited and approved in `11-RESEARCH.md`'s Package
Legitimacy Audit as a required transitive dependency of `lerobot`'s own
`async` extra. No separate checkpoint needed for this install.

**Why the pins (live session, 2026-10-18).** Colab now ships `transformers` 5.13.1.
`transformers` >= 5.9 removed the `cache_position` argument from `create_causal_mask`, and
lerobot 0.6.1's `pi_gemma` still passes it, so pi0 and pi05 fail every inference on Colab's
default. The pin cell installs `transformers==5.5.4` after lerobot, and the version cells
before and after the installs prove it: they fail if `transformers` is not 5.5.4, if
`lerobot` is not 0.6.1, or if pip changed `torch`. `torch` must stay Colab's
2.11.0+cu128.

In [ ]:
# Record torch + transformers before any install (fresh interpreter, not this kernel's
# imports). The versions cell after the installs fails if pip changed torch.
import subprocess
import sys
from pathlib import Path

TORCH_BEFORE = Path('/content/torch_version_before.txt')
probe = "import torch, transformers; print(torch.__version__); print(transformers.__version__)"
torch_version, transformers_version = subprocess.run(
    [sys.executable, '-c', probe], capture_output=True, text=True, check=True).stdout.split()[-2:]
print('before installs: torch', torch_version, '| transformers', transformers_version)
if TORCH_BEFORE.exists():
    print(f'Keeping the first recorded torch {TORCH_BEFORE.read_text().strip()} ({TORCH_BEFORE})')
else:
    TORCH_BEFORE.write_text(torch_version + '\n')
    print(f'Recorded torch {torch_version} → {TORCH_BEFORE}')

In [ ]:
# lerobot 0.6.1 with the `async` extra(s), the dataset extra and num2words.
!pip install 'lerobot[async]==0.6.1'
!pip install 'lerobot[dataset]==0.6.1'
!pip install num2words

In [ ]:
# Pin transformers AFTER lerobot: Colab's 5.13.1 breaks lerobot 0.6.1 (see above).
!pip install 'transformers==5.5.4'

`lerobot`'s installs (the `lerobot[async]` and `lerobot[dataset]` installs above) pull in a `torchaudio` build whose CUDA version doesn't always match Colab's preinstalled `torch` -- confirmed live (Phase 12 Test 7 UAT) to crash the launch cell's `from lerobot.async_inference.policy_server import serve` with `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions`, which cascades into a `ModuleNotFoundError` for `transformers.AutoProcessor` and can leave the kernel needing a full restart. `torchaudio` isn't actually used by anything in this notebook -- uninstalling it removes the mismatched import path entirely (`transformers`' own `is_torchaudio_available()` check then correctly skips it).

In [ ]:
!pip uninstall -y torchaudio

In [ ]:
# Fresh interpreter: this kernel may still hold pre-install packages in memory.
import json
import subprocess
import sys
from pathlib import Path

EXPECTED = {'transformers': '5.5.4', 'lerobot': '0.6.1'}
VERIFIED_TORCH = '2.11.0+cu128'
probe = (
    "import importlib.metadata as m, json, torch\n"
    "def v(p):\n"
    "    try:\n"
    "        return m.version(p)\n"
    "    except m.PackageNotFoundError:\n"
    "        return None\n"
    "print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda,\n"
    "                  'cudnn': torch.backends.cudnn.version(), 'transformers': v('transformers'),\n"
    "                  'lerobot': v('lerobot'), 'diffusers': v('diffusers'), 'grpcio': v('grpcio')}))\n"
)
result = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'version probe failed:\n{result.stderr[-3000:]}')
versions = json.loads(result.stdout.strip().splitlines()[-1])
for name, version in versions.items():
    print(f'{name:13s} {version}')
problems = [f'{name} is {versions[name]}, expected {want}'
            for name, want in EXPECTED.items() if versions[name] != want]
torch_before = Path('/content/torch_version_before.txt').read_text().strip()
if versions['torch'] != torch_before:
    problems.append(f'pip changed torch: {torch_before} → {versions["torch"]}; '
                    'start a fresh runtime and run the cells in order')
if problems:
    raise RuntimeError('; '.join(problems))
if versions['torch'] != VERIFIED_TORCH:
    print(f'WARNING: torch is {versions["torch"]}, but the pins were verified live with '
          f'{VERIFIED_TORCH} on 2026-10-18. Expect surprises.')
print('Versions OK.')

## pi0: serve a local copy with torch.compile off

The pi0 checkpoint ships `compile_model: true` with `compile_mode: max-autotune`. In the
2026-10-18 live session that meant a 465 s first inference, then
`beginAllocateToPool: already recording to mempool_id` after every client reconnect. The next
cell downloads the checkpoint at `backend_configs/pi0.json`'s verified revision into
`/content/ckpt/pi0_so101_nocompile`, sets `compile_model` to false in its `config.json`, and
prints the local smoke command, which passes that local path as `--checkpoint` (the server
loads it from this runtime's disk). pi05's checkpoint already has `compile_model: false`.

In [ ]:
import json
from pathlib import Path

from huggingface_hub import snapshot_download

PI0_REPO = 'amindt/pi0_so101_pickandplace_redblue'
PI0_REVISION = '6d052c61e84eec978070c361d7f86e7d71e08b4f'  # backend_configs/pi0.json verified_revision
PI0_LOCAL = Path('/content/ckpt/pi0_so101_nocompile')
snapshot_download(PI0_REPO, revision=PI0_REVISION, local_dir=str(PI0_LOCAL))
config_path = PI0_LOCAL / 'config.json'
config = json.loads(config_path.read_text())
before = (config.get('compile_model'), config.get('compile_mode'))
config['compile_model'] = False
config_path.write_text(json.dumps(config, indent=4) + '\n')
after = (config.get('compile_model'), config.get('compile_mode'))
print(f'compile_model/compile_mode: {before} → {after} in {config_path}')
print('Local smoke command (control/; the tunnel cell fills in the address):')
print('.venv/bin/python -m vla_bridge.backend_smoke --backend pi0 --server-address <ngrok_tcp_host:port> --checkpoint /content/ckpt/pi0_so101_nocompile --wait-s 300')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `lerobot.async_inference.policy_server`, `diffusion_policy_server.py`, `groot_n17_server.py`, and whatever listens on port 5173.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/policy_server_pi0.log`).
It only succeeds once **its own** still-running process owns port 5173.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report, start_server, stop_servers, wait_until_ready

PORT = 5173
POLICY_SERVER_LOG = '/content/policy_server_pi0.log'
STOP_PATTERNS = ['lerobot.async_inference.policy_server', 'diffusion_policy_server.py', 'groot_n17_server.py']

# The PolicyServer runs as a separate OS subprocess, not a thread in this kernel (Test 7 UAT,
# Phase 12, 2026-09-27):
#   - a crash in it cannot take this kernel (or the tunnel) down
#   - its stdout/stderr go straight to a file via the OS, independent of
#     Jupyter's per-cell output capture, so the log survives even a hard crash
#   - it re-imports torch fresh from disk at subprocess start
#   - CUDA_LAUNCH_BLOCKING=1 forces synchronous CUDA execution, so an async
#     driver-level crash surfaces as a normal Python traceback in the log
#     instead of an opaque process death
#   - KMP_DUPLICATE_LIB_OK / OMP_NUM_THREADS (also set globally above) guard
#     this subprocess too against the "OMP: Error #15" duplicate-runtime crash
# A hung server from an earlier attempt keeps holding GPU memory, which can itself cause
# the next model load to OOM, so stop_servers() clears every known server first.

stop_servers(PORT, STOP_PATTERNS)
print('--- GPU memory before starting the server ---')
gpu_report()
policy_server_process = start_server(
    [sys.executable, '-u', '-m', 'lerobot.async_inference.policy_server', '--host', '0.0.0.0', '--port', '5173'],
    POLICY_SERVER_LOG,
    env_overrides={'KMP_DUPLICATE_LIB_OK': 'TRUE', 'OMP_NUM_THREADS': '1', 'CUDA_LAUNCH_BLOCKING': '1'},
)
health = wait_until_ready(
    policy_server_process, PORT, POLICY_SERVER_LOG,
    timeout_s=90,
)
print(f'Tail it live from another cell with: !tail -f {POLICY_SERVER_LOG}')

## Open a TCP tunnel (not HTTP) to port 5173

In [ ]:
from pyngrok import ngrok

# Explicitly request a TCP tunnel — gRPC needs HTTP/2 semantics that a
# default HTTP tunnel silently breaks (Pitfall 5 in 11-RESEARCH.md).
tunnel = ngrok.connect(5173, "tcp")
print(f"PolicyServer reachable at: {tunnel.public_url}")
# public_url looks like tcp://0.tcp.ngrok.io:12345 — strip the "tcp://"
# prefix; the local RobotClient's --server_address flag wants host:port.
SERVER_ADDRESS = tunnel.public_url.removeprefix("tcp://")
print(f"--server-address {SERVER_ADDRESS}")
SMOKE = '.venv/bin/python -m vla_bridge.backend_smoke --backend pi0 --server-address <ngrok_tcp_host:port> --checkpoint /content/ckpt/pi0_so101_nocompile --wait-s 300'
EPISODE = '.venv/bin/python run_vla_episode.py --backend pi0 --server-address <ngrok_tcp_host:port> --checkpoint /content/ckpt/pi0_so101_nocompile --out outputs/pi0_UNIQUE_NAME --max-steps 20'
for placeholder in ('<ngrok_tcp_host:port>', '<ngrok_tcp_host>:<ngrok_tcp_port>'):
    SMOKE = SMOKE.replace(placeholder, SERVER_ADDRESS)
    EPISODE = EPISODE.replace(placeholder, SERVER_ADDRESS)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

Copy the printed `host:port` (the `tcp://` prefix is already stripped) for the local commands below.

If ngrok's free tier no longer supports a TCP tunnel endpoint by the time
this is run (Assumption A4 in `11-RESEARCH.md`), fall back to `cloudflared`
with a configured Zero Trust tunnel for raw TCP forwarding (heavier setup
than ngrok TCP — cloudflared's zero-config "quick tunnels" are HTTP-only
and will not work here).

## Run locally

After the tunnel cell, run these in `control/`. The tunnel cell prints both commands with the
address filled in. The smoke run uses synthetic hardware (the arm cannot move):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend pi0 --server-address <ngrok_tcp_host:port> --checkpoint /content/ckpt/pi0_so101_nocompile --wait-s 300
```

Then the episode:

```bash
.venv/bin/python run_vla_episode.py --backend pi0 \
  --server-address <ngrok_tcp_host:port> --checkpoint /content/ckpt/pi0_so101_nocompile \
  --out outputs/pi0_UNIQUE_NAME --max-steps 20
```

A healthy server log shows `Policy type: pi0` and then
`Preprocessing and inference took ...s, action shape: ...`-style lines. The first chunk can
take much longer than later ones (model download and load happen at the first handshake).
Failures look different:
- An error about `create_causal_mask` and `cache_position`: `transformers` is not 5.5.4; re-run the pin and versions cells on a fresh runtime.
- `beginAllocateToPool: already recording to mempool_id`: the server loaded a checkpoint with `compile_model: true`; pass `--checkpoint /content/ckpt/pi0_so101_nocompile`.
- A 401/403 or "gated repo" error for `google/paligemma-3b-pt-224`: accept its licence first.

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/policy_server_pi0.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `lerobot.async_inference.policy_server` owns port 5173.
- Package versions, GPU memory.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/policy_server_pi0.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(5173, 'lerobot.async_inference.policy_server', globals().get('policy_server_process'))

In [ ]:
!pip show torch transformers lerobot diffusers | grep -E "^(Name|Version)"
!python -c "import torch; print('torch', torch.__version__); print('cudnn', torch.backends.cudnn.version())"

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted `PolicyServer` is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 5173 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# Teardown: every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'tunnel' in globals():
    try:
        ngrok.disconnect(tunnel.public_url)
        print(f'Disconnected {tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('policy_server_process'), 5173, ['lerobot.async_inference.policy_server', 'diffusion_policy_server.py', 'groot_n17_server.py'])